In [1]:
import pandas as pd
import numpy as np
import os
from config_paths import USE_TEST_DATA, DATA_FOLDER
from config_variables import CATEGORICAL_VARS, CONTINUOUS_VARS, VARIABLES, XWAVE_VARS

# CONFIGURATION
PRIMARY_WAVE = "o"
BACKUP_WAVES = [] if USE_TEST_DATA else ["n", "m", "l", "k"]
WAVE_PICKLE_DIR = f"../{DATA_FOLDER}/2_pickle_ukhls_waves"
OUTPUT_DIR  = f"../{DATA_FOLDER}/3_backfill_ukhls_waves"
OUTPUT_FILE = os.path.join(OUTPUT_DIR, f"{PRIMARY_WAVE}_indresp_backfilled.pkl")

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Targeting Wave {PRIMARY_WAVE} with backups from {BACKUP_WAVES}")


def safe_numeric(series):
    numeric = pd.to_numeric(series, errors='coerce')
    if pd.api.types.is_numeric_dtype(numeric):
        numeric = numeric.replace([np.inf, -np.inf], np.nan)
    return numeric


def get_base_code(col_name, wave_prefix):
    """Strip wave prefix to get the base variable code (e.g. 'o_age_dv' -> 'age_dv')."""
    prefix = f"{wave_prefix}_"
    return col_name[len(prefix):] if col_name.startswith(prefix) else col_name


def resolve_backfill_column(V, trigger_values):
    """
    V: (n_rows, n_waves) float, columns [o, n, m, l, k, ...].
    trigger_values: list of values that should trigger a look-back (e.g. [-9, -7, -2, -1]).
    NaN in wave o always triggers a look-back.
    For triggered rows, scan older waves for the first finite value >= 0;
    if none found, keep the original wave o value.
    """
    _, W = V.shape
    if W == 0:
        return np.full(V.shape[0], np.nan, dtype=np.float64)

    v0 = V[:, 0].astype(np.float64, copy=True)
    result = np.copy(v0)

    trigger_set = np.array(trigger_values, dtype=np.float64)
    is_trigger = np.isnan(v0)
    for tv in trigger_set:
        is_trigger |= np.isclose(v0, tv, rtol=0, atol=0)

    if not is_trigger.any() or W <= 1:
        return result

    first_hist = np.full(V.shape[0], np.nan, dtype=np.float64)
    for j in range(1, W):
        colj = V[:, j]
        hit = is_trigger & ~np.isfinite(first_hist) & np.isfinite(colj) & (colj >= 0)
        first_hist[hit] = colj[hit]

    got_hist = np.isfinite(first_hist)
    result[is_trigger & got_hist] = first_hist[is_trigger & got_hist]
    return result


def load_renamed_wave(wave):
    path = os.path.join(WAVE_PICKLE_DIR, f"{wave}_indresp_optimized.pkl")
    if not os.path.exists(path):
        return None
    df = pd.read_pickle(path).set_index("pidp")
    if wave != PRIMARY_WAVE:
        df.columns = df.columns.str.replace(f"{wave}_", f"{PRIMARY_WAVE}_", regex=False)
    return df


def stack_column_numeric(col, master_index, df_by_wave):
    cols = []
    for _, dfw in df_by_wave:
        if dfw is None:
            cols.append(np.full(len(master_index), np.nan, dtype=np.float64))
            continue
        if col not in dfw.columns:
            cols.append(np.full(len(master_index), np.nan, dtype=np.float64))
            continue
        s = dfw[col].reindex(master_index)
        if isinstance(s.dtype, pd.CategoricalDtype):
            s = s.astype("object")
        cols.append(pd.to_numeric(s, errors="coerce").to_numpy(dtype=np.float64))
    if not cols:
        return np.empty((len(master_index), 0))
    return np.column_stack(cols)


def build_expanded_master():
    primary_file = os.path.join(WAVE_PICKLE_DIR, f"{PRIMARY_WAVE}_indresp_optimized.pkl")
    if not os.path.exists(primary_file):
        raise FileNotFoundError(f"Could not find primary wave file: {primary_file}")

    print(f"Loading Primary Wave ({PRIMARY_WAVE})...")
    df_o = pd.read_pickle(primary_file).set_index("pidp")

    df_by_wave = [("o", df_o)]
    for w in BACKUP_WAVES:
        df_b = load_renamed_wave(w)
        if df_b is None:
            print(f"Warning: {w} wave pickle not found. Skipping wave {w}.")
            df_by_wave.append((w, None))
        else:
            df_by_wave.append((w, df_b))

    # Only respondents present in wave o; do not append people who lack an o interview.
    df_master = df_o.copy()

    # Collect all backfill-eligible columns across every wave (not just wave o)
    all_bf_cols = set()
    for _, dfw in df_by_wave:
        if dfw is None:
            continue
        for c in dfw.columns:
            base = get_base_code(c, PRIMARY_WAVE)
            bf = VARIABLES.get(base, {}).get("backfill")
            if bf and isinstance(bf, list) and len(bf) > 0:
                prefixed = f"{PRIMARY_WAVE}_{base}"
                all_bf_cols.add(prefixed)

    print("Resolving per-variable backfill (VARIABLES[base]['backfill'])...")
    n_bf_cols = 0
    processed = set()
    for col in list(df_master.columns) + sorted(all_bf_cols - set(df_master.columns)):
        if col in processed or "idp" in col.lower():
            continue
        processed.add(col)
        base = get_base_code(col, PRIMARY_WAVE)
        bf = VARIABLES.get(base, {}).get("backfill")

        if bf and isinstance(bf, list) and len(bf) > 0:
            V = stack_column_numeric(col, df_master.index, df_by_wave)
            if V.shape[1] == 0:
                continue
            df_master[col] = resolve_backfill_column(V, bf)
            n_bf_cols += 1
        else:
            if col not in df_o.columns:
                continue
            needs_obj = isinstance(df_master[col].dtype, pd.CategoricalDtype)
            if isinstance(df_o[col].dtype, pd.CategoricalDtype):
                needs_obj = True
            if needs_obj:
                df_master[col] = df_master[col].astype("object")
            df_master[col] = df_o[col]

    print(f"   -> Cross-wave resolution on {n_bf_cols} columns; wave-o only for backfill=False.")

    # ── xwavedat variables (no backfill — authoritative source) ─────────────
    if XWAVE_VARS:
        xwave_file = os.path.join(WAVE_PICKLE_DIR, "xwavedat.pkl")
        if os.path.exists(xwave_file):
            df_xwave = pd.read_pickle(xwave_file).set_index("pidp")
            print(f"\nLoading xwavedat variables ({len(XWAVE_VARS)} variable(s))...")
            for base in sorted(XWAVE_VARS):
                target_col = f"{PRIMARY_WAVE}_{base}"
                if base not in df_xwave.columns:
                    print(f"  WARNING: {base} not found in xwavedat — skipping")
                    continue
                vals = pd.to_numeric(df_xwave[base].reindex(df_master.index), errors="coerce")
                df_master[target_col] = vals
                n_valid = int(vals.notna().sum())
                print(f"  {target_col} <- xwavedat.{base}  ({n_valid:,} valid)")
        else:
            print(f"\nWARNING: xwavedat.pkl not found at {xwave_file} — skipping xwave variables.")

    print("\nNaN after backfill (column name, value, count, sample pidp):")
    any_nan = False
    for col in df_master.columns:
        if "idp" in col.lower():
            continue
        nan_mask = df_master[col].isna()
        if not nan_mask.any():
            continue
        any_nan = True
        n_nan = int(nan_mask.sum())
        sample_pidp = df_master.index[nan_mask][:10].tolist()
        print(f"  {col}: value=nan  n={n_nan:,}  sample_pidp={sample_pidp}")
    if not any_nan:
        print("  (no NaNs)")

    df_master = df_master.reset_index()

    print("Optimizing dtypes...")
    for col in df_master.columns:
        if "idp" in col.lower():
            id_numeric = safe_numeric(df_master[col]).fillna(0)
            df_master[col] = id_numeric.astype(np.int64)
            continue

        base = get_base_code(col, PRIMARY_WAVE)

        if base in CATEGORICAL_VARS:
            df_master[col] = df_master[col].astype("category")
        elif base in CONTINUOUS_VARS:
            df_master[col] = pd.to_numeric(safe_numeric(df_master[col]), downcast="float")
        elif pd.api.types.is_float_dtype(df_master[col]):
            df_master[col] = pd.to_numeric(safe_numeric(df_master[col]), downcast="float")
        elif pd.api.types.is_integer_dtype(df_master[col]):
            df_master[col] = safe_numeric(df_master[col]).astype("Int64")

    df_master.to_pickle(OUTPUT_FILE, protocol=5)
    print(f"DONE. Final Master size: {len(df_master):,} rows.")
    print(f"File saved to: {OUTPUT_FILE}")


build_expanded_master()



Targeting Wave o with backups from ['n', 'm', 'l', 'k']
Loading Primary Wave (o)...
Resolving per-variable backfill (VARIABLES[base]['backfill'])...
   -> Cross-wave resolution on 13 columns; wave-o only for backfill=False.

Loading xwavedat variables (3 variable(s))...
  o_doby_dv <- xwavedat.doby_dv  (32,849 valid)
  o_racel_dv <- xwavedat.racel_dv  (32,849 valid)
  o_sex_dv <- xwavedat.sex_dv  (32,849 valid)

NaN after backfill (column name, value, count, sample pidp):
  o_englang: value=nan  n=26,875  sample_pidp=[22445, 29925, 76165, 280165, 469205, 599765, 732365, 1587125, 2888645, 3062725]
  o_hiquao_dv: value=nan  n=10,750  sample_pidp=[3062725, 3424485, 68014291, 68014295, 68014299, 68060535, 68089168, 68105411, 68112207, 68141455]
  o_locserc: value=nan  n=10,682  sample_pidp=[3062725, 68014291, 68014295, 68014299, 68060535, 68089168, 68105411, 68112207, 68141455, 68180899]
  o_locserd: value=nan  n=10,628  sample_pidp=[3062725, 68014291, 68014295, 68014299, 68060535, 6808916